# 05 - H\u2082 Ground State with QONTOS VQE

The **Variational Quantum Eigensolver (VQE)** is a hybrid quantum-classical
algorithm for finding the ground-state energy of molecular Hamiltonians.
It is one of the most promising near-term applications of quantum computing.

This notebook demonstrates:

1. Building an H\u2082 ansatz circuit
2. Submitting it through the QONTOS pipeline
3. Extracting energy estimates from measurement results
4. Plotting the energy landscape as bond distance varies
5. Connection to quantum chemistry applications

## Environment

This notebook works with the QONTOS local simulator. No quantum hardware
access is needed.

```bash
# pip install git+https://github.com/qontos/qontos.git@v0.2.0 git+https://github.com/qontos/qontos-sim.git@v0.1.0
```

## Prerequisites

- Python >= 3.10
- `qontos` and `qontos-sim` installed
- Basic understanding of variational quantum algorithms
- Familiarity with previous notebooks in this series

In [ ]:
import math
import os

from qontos import QontosClient
from qontos.circuit import CircuitNormalizer

## Background: Variational Quantum Eigensolver

VQE works by:

1. **Preparing** a parameterized quantum state (the *ansatz*)
2. **Measuring** the expectation value of the Hamiltonian
3. **Optimizing** the parameters classically to minimize energy

For H\u2082, a minimal ansatz uses 2 qubits with a single variational
parameter \u03b8 that controls the rotation angle in the ansatz circuit.

## Build the H\u2082 Ansatz Circuit

The UCCSD-inspired ansatz for H\u2082 at minimal basis consists of:
- Initial state preparation (X gate on qubit 0 for |01\u27e9 reference)
- A parameterized excitation operator controlled by angle \u03b8
- Entangling CNOT gates

We write a function that generates the OpenQASM circuit for a given \u03b8.

In [ ]:
def build_h2_ansatz(theta: float) -> str:
    """Build a 2-qubit H2 VQE ansatz circuit in OpenQASM 2.0.
    
    The ansatz prepares a trial state for the H2 molecule using
    a single variational parameter theta.
    
    Args:
        theta: Variational parameter (rotation angle in radians).
    
    Returns:
        OpenQASM 2.0 string for the ansatz circuit.
    """
    return f"""\
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
x q[0];
ry({theta}) q[1];
cx q[1],q[0];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""


# Example: build ansatz at theta = pi/4
example_qasm = build_h2_ansatz(math.pi / 4)
print(example_qasm)

In [ ]:
# Normalize the example circuit to verify it's valid
normalizer = CircuitNormalizer()
ir = normalizer.normalize(input_type="openqasm", source=example_qasm)

print(f"Qubits    : {ir.num_qubits}")
print(f"Gates     : {ir.gate_count}")
print(f"Depth     : {ir.depth}")

## Submit via QONTOS

We submit the ansatz circuit through the standard QONTOS pipeline.
The platform handles normalization, scheduling, execution, and result
aggregation -- the same pipeline used for any quantum workload.

In [ ]:
client = QontosClient(
    base_url=os.getenv("QONTOS_BASE_URL", "http://localhost:8000"),
    api_key=os.getenv("QONTOS_API_KEY", "demo"),
)

# Submit the ansatz at a specific theta
theta_val = math.pi / 4
qasm = build_h2_ansatz(theta_val)

job = client.submit_job(
    circuit=qasm,
    shots=4096,
    name=f"VQE H2 theta={theta_val:.4f}",
    tags={"source": "notebook-05", "algorithm": "vqe", "molecule": "h2"},
)
print(f"Job ID: {job.id}")
print(f"Status: {job.status}")

## Extract Energy Estimate

From the measurement counts, we compute the expectation value of the
Z\u2297Z operator, which maps to the electronic energy of H\u2082.

The mapping is:
- |00\u27e9 and |11\u27e9 contribute +1
- |01\u27e9 and |10\u27e9 contribute -1

In [ ]:
def compute_zz_expectation(counts: dict) -> float:
    """Compute <ZZ> expectation value from measurement counts.
    
    Args:
        counts: Dictionary mapping bitstrings to counts.
    
    Returns:
        Expectation value of ZZ operator.
    """
    total = sum(counts.values())
    expectation = 0.0
    for bitstring, count in counts.items():
        # Parity: +1 for even parity (00, 11), -1 for odd parity (01, 10)
        parity = (-1) ** sum(int(b) for b in bitstring)
        expectation += parity * count
    return expectation / total


# Wait for the job and extract results
job = client.wait_for_job(job.id, timeout=120)

if job.runs:
    result = client.get_results(job.runs[0].id)
    zz_exp = compute_zz_expectation(result.counts)
    print(f"Counts       : {dict(sorted(result.counts.items()))}")
    print(f"<ZZ>         : {zz_exp:.4f}")
    print(f"Total shots  : {result.total_shots}")
else:
    print("No runs returned.")

## Plot the Energy Landscape

To find the ground state, VQE sweeps over \u03b8 values and plots the
energy as a function of the variational parameter. The minimum of
this curve corresponds to the ground-state energy.

Here we sweep \u03b8 from 0 to \u03c0 and collect energy estimates.

In [ ]:
# Sweep theta values
theta_values = [i * math.pi / 8 for i in range(9)]  # 0 to pi in 9 steps
energies = []

for theta in theta_values:
    qasm = build_h2_ansatz(theta)
    job = client.submit_job(
        circuit=qasm,
        shots=2048,
        name=f"VQE sweep theta={theta:.4f}",
        tags={"source": "notebook-05", "sweep": "true"},
    )
    job = client.wait_for_job(job.id, timeout=120)
    
    if job.runs:
        result = client.get_results(job.runs[0].id)
        energy = compute_zz_expectation(result.counts)
    else:
        energy = float("nan")
    
    energies.append(energy)
    print(f"  theta={theta:.4f}  <ZZ>={energy:+.4f}")

# Find the minimum
min_idx = energies.index(min(energies))
print(f"\nMinimum energy: {energies[min_idx]:.4f} at theta={theta_values[min_idx]:.4f}")

In [ ]:
# ASCII plot of the energy landscape
print("\nEnergy Landscape (theta vs <ZZ>):")
print()

# Scale to fit in 40 columns
min_e = min(energies)
max_e = max(energies)
width = 40

for theta, energy in zip(theta_values, energies):
    if max_e > min_e:
        pos = int(width * (energy - min_e) / (max_e - min_e))
    else:
        pos = width // 2
    marker = "*" if energy == min(energies) else "o"
    bar = " " * pos + marker
    print(f"  theta={theta:.2f} |{bar}")

print(f"{'':>14}|{'min':>{width // 2}}{'':>{width // 2 - 3}}max|")

## Connection to Quantum Chemistry

The H\u2082 molecule is a benchmark problem for quantum chemistry on quantum
computers. Key points:

- **Exact ground-state energy** of H\u2082 at equilibrium bond distance is
  approximately -1.137 Hartree
- The VQE approach maps the electronic Hamiltonian to qubit operators
  using transformations like Jordan-Wigner or Bravyi-Kitaev
- QONTOS handles the quantum execution transparently -- the chemistry
  application only needs to provide circuits and process expectation values
- For larger molecules, QONTOS circuit partitioning becomes essential:
  ansatz circuits for molecules like LiH or BeH\u2082 may require more qubits
  than a single module supports

The same QONTOS pipeline used here -- normalize, partition, schedule,
execute, aggregate -- scales from H\u2082 to industrially relevant molecules.

In [ ]:
# Clean up
client.close()
print("Client closed.")

## Summary

In this notebook you learned:

- How to build a parameterized VQE ansatz circuit for H\u2082
- How to submit VQE circuits through the QONTOS pipeline
- How to extract energy estimates from measurement counts
- How to sweep parameters and plot the energy landscape
- How VQE connects to real quantum chemistry applications

**Further reading:**
- [examples/python/async_jobs.py](../examples/python/async_jobs.py) -- async multi-job workflows
- [examples/python/execution_proof.py](../examples/python/execution_proof.py) -- integrity verification